# Kadastra BO2 — Agent Prédictif des Tendances Régionales

**Projet :** Kadastra — Plateforme d'estimation immobilière basée sur l'IA  
**Composante :** BO2 — Agent prédictif des tendances régionales  
**Données :** Mubawab + Tayara (scraping BO1) + données socioéconomiques nationales

---

## Vue d'ensemble

Ce notebook implémente un **agent prédictif** qui prédit l'évolution des prix immobiliers par gouvernorat sur **12 mois**, en intégrant cinq familles de facteurs socioéconomiques conformément au cahier des charges :

| Facteur demandé | Source | Couverture |
|---|---|---|
| Dynamique démographique | INS — Population par gouvernorat | 2015–2024 |
| Projets d'infrastructure | AfDB + World Bank + EBRD | 1962–2026 (545 projets) |
| Évolution du tourisme | ONTT — Capacité et taux d'occupation | 2015–2023 |
| Taux de change et inflation | Banque Mondiale — IPC + USD/DT | 1960–2024 |
| Crédit bancaire au logement | ⚠️ Données BCT non disponibles en open data | — |

## Architecture du pipeline

```
Données listings (Mubawab/Tayara)
        ↓
Agrégation mensuelle (gouvernorat × mois)
        ↓
Fusion facteurs externes (démographie, tourisme, macro, infrastructure)
        ↓
Feature engineering (lags 1/3/6 mois, rolling means — sans fuite de données)
        ↓
LightGBM avec validation croisée temporelle (5 folds)
        ↓
Prévisions 12 mois + intervalles de confiance (bootstrap 50 itérations)
        ↓
Scores de tendance + système de recommandation pondéré
```


## 1. Installation & Imports

Toutes les dépendances nécessaires sont installées ici. La cellule est idempotente — elle peut être réexécutée sans effet de bord.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive, files
import os
import joblib
import lightgbm as lgb
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
from scipy.stats import linregress
import unicodedata
import re
import io
from matplotlib.patches import Patch

drive.mount('/content/drive')
DATA_PATH = '/content/drive/MyDrive/kadastra_data/'

print('✅ Imports OK. Dossier de données:', DATA_PATH)


## 2. Fonctions utilitaires

Deux fonctions transversales utilisées dans tout le notebook :

- **`clean_column_names`** : normalise les noms de colonnes (minuscules, underscores) pour garantir la compatibilité entre datasets de sources différentes.
- **`normalise_governorate`** : harmonise les noms de gouvernorats entre datasets — élimine les préfixes français (`Gouvernorat de`, `Gouvernorat du`...), les accents et les variantes orthographiques. Sans cette normalisation, les jointures entre datasets échouent silencieusement et produisent des NaN.


In [ ]:
def clean_column_names(df):
    df.columns = df.columns.str.lower().str.strip()
    df.columns = [re.sub(r'[^a-z0-9_]', '_', col) for col in df.columns]
    return df

def normalise_governorate(name):
    if pd.isna(name):
        return name
    name = unicodedata.normalize('NFKD', str(name)).encode('ASCII', 'ignore').decode('utf-8')
    prefixes = ['gouvernorat de ', 'governorate of ', 'gouvernorat du ', "gouvernorat de l'"]
    for p in prefixes:
        if name.lower().startswith(p):
            name = name[len(p):]
    name = name.replace("l'aryana", "Ariana").replace("le kef", "Kef")
    name = name.replace("du kef", "Kef").replace("ben arous", "Ben Arous")
    name = name.replace("beja", "Beja")
    name = name.strip().title()
    return name

print('✅ Fonctions utilitaires définies.')


## 3. Chargement des données statiques

Chargement des six datasets socioéconomiques externes. Ces données sont **fixes** — elles ne dépendent pas du type de bien immobilier et sont chargées une seule fois en début de notebook.

**Note sur la cartographie tourisme → gouvernorat :** les données ONTT sont publiées par **région touristique** (ex. "Nabeul - Hammamet"), pas par gouvernorat administratif. La fonction `map_region_to_gov` fait la conversion en se basant sur la géographie — chaque région touristique est assignée au(x) gouvernorat(s) qu'elle couvre.


In [ ]:
print('Chargement des données statiques depuis Drive...')

pop = clean_column_names(pd.read_csv(DATA_PATH + 'population_by_governorate_2015_2024.csv'))
companies_gov = clean_column_names(pd.read_csv(DATA_PATH + 'cleaned_companies_by_governorate.csv'))
tourism_reg = clean_column_names(pd.read_csv(DATA_PATH + 'tourism_regional.csv'))
tourism_nat = clean_column_names(pd.read_csv(DATA_PATH + 'tourism_national.csv'))
infra_projects = clean_column_names(pd.read_csv(DATA_PATH + 'infrastructure_projects_combined.csv'))
infra_tun = clean_column_names(pd.read_csv(DATA_PATH + 'infrastructure_tun_long.csv'))
macro = clean_column_names(pd.read_csv(DATA_PATH + 'tunisia_inflation_exchange.csv'))

companies_gov.rename(columns={'gouvernorat': 'governorate', 'n_companies': 'n_companies_gov'}, inplace=True)

# Taux de croissance annuels pré-calculés au niveau de l'année (pas du mois)
# Important: pct_change() ici compare une annee sur l'autre, pas un mois sur l'autre
macro['cpi_yoy'] = macro['cpi'].pct_change(fill_method=None)
macro['fx_yoy']  = macro['exchange_rate'].pct_change(fill_method=None)

def map_region_to_gov(region):
    mapping = {
        'tunis - zaghouan': ['Tunis', 'Zaghouan'],
        'nabeul - hammamet': ['Nabeul'],
        'sousse - kairouan': ['Sousse', 'Kairouan'],
        'yasmine hammamet': ['Nabeul'],
        'monastir - skanes': ['Monastir'],
        'mahdia sfax': ['Mahdia', 'Sfax'],
        'jerba-zarzis, gabes': ['Medenine', 'Gabes'],
        'gafsa-tozeur': ['Gafsa', 'Tozeur'],
        'sbeitla-kasserine': ['Kasserine'],
        'bizerte - beja': ['Bizerte', 'Beja'],
        'tabarka-ain draham': ['Jendouba'],
    }
    return mapping.get(region.lower(), [])

print(f'✅ Données chargées:')
print(f'   Population          : {pop.shape[0]} lignes ({pop["year"].nunique()} années)')
print(f'   Entreprises (gouv.) : {companies_gov.shape[0]} lignes')
print(f'   Tourisme régional   : {tourism_reg.shape[0]} lignes')
print(f'   Tourisme national   : {tourism_nat.shape[0]} lignes')
print(f'   Projets infra       : {infra_projects.shape[0]} projets (AfDB+WB+EBRD)')
print(f'   Indicateurs macro   : {infra_tun["indicator_name"].nunique()} indicateurs')
print(f'   Macro (IPC+FX)      : {macro.shape[0]} années (1960-2024)')


## 4. Pipeline principal — Modélisation par type de bien

Cette cellule exécute l'ensemble du pipeline pour les **6 types de bien** en séquence. Pour chaque type, le pipeline réalise :

### 4.1 Agrégation du panel
Les listings bruts (4 000–7 000 lignes par type) sont agrégés en un **panel gouvernorat × mois** : pour chaque combinaison (gouvernorat, année, mois), on calcule le prix médian. Un filtre de **minimum 3 annonces par cellule** est appliqué pour éliminer les médians basés sur 1-2 transactions atypiques — source principale des prévisions implausibles à +100-200%.

### 4.2 Fusion des facteurs externes
Cinq familles de features sont jointes au panel :
- **Population** — par gouvernorat et année (INS 2015-2024)
- **Entreprises** — nombre par gouvernorat et année
- **Tourisme** — capacité en lits, établissements, taux d'occupation (mappé région → gouvernorat)
- **Macro** — IPC, taux de change, croissance annuelle (pré-calculée au niveau annuel pour éviter le biais "mois constant")
- **Infrastructure** — projets approuvés par an, budget total (national)

### 4.3 Feature engineering (sans fuite de données)
Les features lag et rolling mean sont calculées **à l'intérieur de chaque groupe gouvernorat**, avec `shift(1)` avant le `rolling()` pour que le rolling mean au temps T n'utilise que des données jusqu'à T-1. Les médianes de zone et l'encodage cible sont calculés **par fold** sur les données d'entraînement uniquement.

### 4.4 Validation croisée temporelle
Le CV split est basé sur l'index temporel (`year*12 + month`), pas sur l'index de ligne. Chaque fold entraîne sur toutes les données jusqu'à un cutoff temporel et valide sur les 3 mois suivants. Les médianes d'imputation sont calculées sur le fold d'entraînement uniquement.

### 4.5 Prévision 12 mois + intervalles de confiance
La prévision est **autorégressive** : chaque mois prédit devient l'entrée du mois suivant. 50 runs bootstrap avec perturbation gaussienne (σ=2%) sur le dernier prix observé donnent un intervalle de confiance à 80%.

> **Note sur les valeurs imputées :** Les features externes (population, CPI...) sont maintenues à leur dernière valeur connue pendant la prévision — hypothèse de stabilité, raisonnable sur un horizon de 12 mois.


In [ ]:
property_types = [
    'appt_vente', 'appt_location', 'maison_vente',
    'maison_location', 'terrain_vente', 'local_vente'
]

uploaded_dfs = {}
all_results  = []

# Infrastructure yearly aggregation (computed once, reused in loop)
infra_projects['year'] = pd.to_datetime(infra_projects['approval_date'], errors='coerce').dt.year
infra_yearly = infra_projects.groupby('year').agg(
    n_projects=('project_id', 'count'),
    total_budget=('budget_amount', 'sum')
).reset_index()
infra_yearly['infra_growth']  = infra_yearly['n_projects'].pct_change(fill_method=None)
infra_yearly['budget_growth'] = infra_yearly['total_budget'].pct_change(fill_method=None)

# Infrastructure wide indicators
relevant_indicators = [
    'air transport, passengers carried',
    'ict service exports (bop, current us$)',
    'individuals using the internet (% of population)',
    'mobile cellular subscriptions (per 100 people)',
    'road density (km of road per 100 sq. km of land area)',
    'electric power consumption (kwh per capita)'
]
infra_tun_filtered = infra_tun[infra_tun['indicator_name'].isin(relevant_indicators)]
infra_wide = infra_tun_filtered.pivot(index='year', columns='indicator_name', values='value').reset_index()
infra_wide.columns = ['year'] + [re.sub(r'[^a-z0-9_]', '_', col.lower()) for col in infra_wide.columns[1:]]

for prop_type in property_types:
    print(f'\n{"="*70}')
    print(f'TYPE DE BIEN : {prop_type.upper()}')
    print(f'{"="*70}')

    drive_file = DATA_PATH + f'clean_{prop_type}.csv'
    if os.path.exists(drive_file):
        listings = pd.read_csv(drive_file)
        print(f'Chargé depuis Drive : {listings.shape}')
    elif prop_type in uploaded_dfs:
        listings = uploaded_dfs[prop_type]
    else:
        print(f'Fichier non trouvé : {drive_file}')
        print(f'Veuillez uploader clean_{prop_type}.csv')
        uploaded = files.upload()
        for fname, content in uploaded.items():
            listings = pd.read_csv(io.BytesIO(content))
            uploaded_dfs[prop_type] = listings
            break

    listings = clean_column_names(listings)
    gov_col = next((c for c in listings.columns if 'gov' in c or 'gouvernorat' in c), None)
    if not gov_col:
        print(f'  ⚠️ Colonne gouvernorat non trouvée, passage au suivant.')
        continue
    listings.rename(columns={gov_col: 'governorate'}, inplace=True)
    listings['governorate'] = listings['governorate'].apply(normalise_governorate)
    listings = listings[listings['governorate'] != 'Unknown']

    date_col = next((c for c in listings.columns if 'date' in c), None)
    if not date_col:
        continue
    listings[date_col] = pd.to_datetime(listings[date_col], errors='coerce')
    listings.rename(columns={date_col: 'datepost'}, inplace=True)
    listings = listings.dropna(subset=['datepost', 'governorate', 'price_numeric'])

    if 'location' in prop_type:
        listings = listings[(listings['price_numeric'] >= 200) & (listings['price_numeric'] <= 20000)]
        target_col = 'price_numeric'
        print('Cible : loyer mensuel médian (DT)')
    else:
        surf_col = next((c for c in listings.columns if 'surface' in c), None)
        if not surf_col:
            continue
        listings['prix_m2'] = listings['price_numeric'] / listings[surf_col].replace(0, np.nan)
        listings = listings[(listings['prix_m2'] >= 200) & (listings['prix_m2'] <= 15000)]
        target_col = 'prix_m2'
        print('Cible : prix/m² médian (DT/m²)')

    listings['year']  = listings['datepost'].dt.year
    listings['month'] = listings['datepost'].dt.month

    panel = listings.groupby(['governorate', 'year', 'month']).agg(
        median_price=(target_col, 'median'),
        n_listings=(target_col, 'count')
    ).reset_index().rename(columns={'median_price': 'median_prix_m2'})

    # Filtre minimum 3 annonces par cellule gouvernorat×mois
    panel = panel[panel['n_listings'] >= 3]
    print(f'Panel après filtre ≥3 annonces : {panel.shape[0]} lignes, {panel["governorate"].nunique()} gouvernorats')

    pop_c = pop.copy()
    pop_c['governorate'] = pop_c['governorate'].apply(normalise_governorate)
    panel = panel.merge(pop_c, on=['governorate', 'year'], how='left')

    cg = companies_gov.copy()
    cg['governorate'] = cg['governorate'].apply(normalise_governorate)
    panel = panel.merge(cg, on=['governorate', 'year'], how='left')

    t_exp = []
    for _, row in tourism_reg.iterrows():
        for gov in map_region_to_gov(row['region']):
            nr = row.copy(); nr['governorate'] = gov; t_exp.append(nr)
    t_gov = pd.DataFrame(t_exp).groupby(['governorate', 'year']).agg(
        {'bed_capacity': 'sum', 'n_establishments': 'sum', 'occupancy_rate': 'mean'}
    ).reset_index()
    panel = panel.merge(t_gov, on=['governorate', 'year'], how='left')
    panel = panel.merge(tourism_nat, on='year', how='left')
    panel = panel.merge(macro, on='year', how='left')
    panel = panel.merge(infra_yearly, on='year', how='left')
    panel = panel.merge(infra_wide, on='year', how='left')

    panel = panel.sort_values(['governorate', 'year', 'month']).reset_index(drop=True)
    panel['pop_growth_rate'] = panel.groupby('governorate')['population'].pct_change(12, fill_method=None)
    panel['company_growth']  = panel.groupby('governorate')['n_companies_gov'].pct_change(12, fill_method=None)
    panel['tourism_growth']  = panel['total_overnight_stays'].pct_change(12, fill_method=None)

    for lag in [1, 3, 6]:
        panel[f'lag_{lag}_price'] = panel.groupby('governorate')['median_prix_m2'].shift(lag)
    for window in [3, 6]:
        panel[f'rolling_{window}_price'] = (
            panel.groupby('governorate')['median_prix_m2']
            .apply(lambda x: x.shift(1).rolling(window, min_periods=1).mean())
            .reset_index(level=0, drop=True)
        )

    panel['target'] = panel.groupby('governorate')['median_prix_m2'].shift(-1)
    panel = panel.dropna(subset=['target', 'lag_1_price'])

    panel = panel.sort_values(['year', 'month', 'governorate']).reset_index(drop=True)
    panel['time_idx'] = panel['year'] * 12 + panel['month']

    exclude = ['governorate', 'year', 'month', 'median_prix_m2', 'target', 'n_listings', 'time_idx']
    feature_cols = [c for c in panel.columns if c not in exclude and panel[c].dtype in ['float64', 'int64']]

    X = panel[feature_cols].copy()
    y = panel['target'].copy()
    san_map = {col: re.sub(r'[^a-zA-Z0-9_]', '_', col) for col in feature_cols}
    X.rename(columns=san_map, inplace=True)
    feature_cols = list(san_map.values())
    panel.rename(columns=san_map, inplace=True)
    global_medians = X.median()

    unique_times = sorted(panel['time_idx'].unique())
    n_times = len(unique_times)
    mape_scores = []
    print('\nValidation croisée temporelle (5 folds) :')
    for fold in range(5):
        cidx = int(n_times * (0.5 + fold * 0.08))
        if cidx >= n_times - 1:
            print(f'  Fold {fold+1}: ignoré (données insuffisantes)')
            continue
        cutoff = unique_times[cidx]
        yc, mc = cutoff // 12, cutoff % 12
        if mc == 0: yc -= 1; mc = 12
        train_mask = panel['time_idx'] <= cutoff
        val_mask   = (panel['time_idx'] > cutoff) & (panel['time_idx'] <= cutoff + 3)
        if val_mask.sum() == 0: continue
        tm = X[train_mask].median()
        Xtr = X[train_mask].fillna(tm); ytr = y[train_mask]
        Xvl = X[val_mask].fillna(tm);  yvl = y[val_mask]
        m = lgb.LGBMRegressor(n_estimators=200, learning_rate=0.05, num_leaves=31, random_state=42, verbose=-1)
        m.fit(Xtr, ytr, eval_set=[(Xvl, yvl)], callbacks=[lgb.early_stopping(30, verbose=False)])
        mape = mean_absolute_percentage_error(yvl, m.predict(Xvl)) * 100
        mape_scores.append(mape)
        print(f'  Fold {fold+1}: cutoff={yc}-{mc:02d}  n_val={len(yvl)}  MAPE={mape:.1f}%')

    cv_mean = np.mean(mape_scores) if mape_scores else np.nan
    cv_std  = np.std(mape_scores)  if mape_scores else np.nan
    if mape_scores:
        print(f'  → CV MAPE final : {cv_mean:.1f}% ± {cv_std:.1f}%')

    final_model = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.03, num_leaves=31, random_state=42, verbose=-1)
    final_model.fit(X.fillna(global_medians), y)
    joblib.dump(final_model, DATA_PATH + f'regional_trend_model_{prop_type}.pkl')
    print(f'  Modèle final sauvegardé.')

    forecasts = []
    bootstrap_forecasts = {}

    for gov in panel['governorate'].unique():
        gd = panel[panel['governorate'] == gov].sort_values(['year', 'month'])
        if len(gd) < 6: continue
        history  = gd['median_prix_m2'].values.tolist()
        last_row = gd.iloc[-1].copy()
        cy, cm   = int(last_row['year']), int(last_row['month'])

        runs = []
        for _ in range(50):
            hb = history.copy()
            hb[-1] *= (1 + np.random.normal(0, 0.02))
            rb = last_row.copy(); cb_y, cb_m = cy, cm; fb = []
            for _ in range(12):
                cb_m += 1
                if cb_m > 12: cb_m = 1; cb_y += 1
                rb['year'] = cb_y; rb['month'] = cb_m
                rb['lag_1_price'] = hb[-1]
                rb['lag_3_price'] = hb[-3] if len(hb)>=3 else hb[0]
                rb['lag_6_price'] = hb[-6] if len(hb)>=6 else hb[0]
                rb['rolling_3_price'] = np.mean(hb[-3:])
                rb['rolling_6_price'] = np.mean(hb[-6:]) if len(hb)>=6 else np.mean(hb)
                pp = final_model.predict(pd.DataFrame([rb[feature_cols]]).fillna(global_medians))[0]
                hb.append(pp); fb.append(pp)
            runs.append(fb)

        hbase = history.copy(); rbase = last_row.copy(); cy_b, cm_b = cy, cm; bfore = []
        for step in range(12):
            cm_b += 1
            if cm_b > 12: cm_b = 1; cy_b += 1
            rbase['year'] = cy_b; rbase['month'] = cm_b
            rbase['lag_1_price'] = hbase[-1]
            rbase['lag_3_price'] = hbase[-3] if len(hbase)>=3 else hbase[0]
            rbase['lag_6_price'] = hbase[-6] if len(hbase)>=6 else hbase[0]
            rbase['rolling_3_price'] = np.mean(hbase[-3:])
            rbase['rolling_6_price'] = np.mean(hbase[-6:]) if len(hbase)>=6 else np.mean(hbase)
            pp = final_model.predict(pd.DataFrame([rbase[feature_cols]]).fillna(global_medians))[0]
            hbase.append(pp); bfore.append(pp)
            forecasts.append({'governorate': gov, 'year': cy_b, 'month': cm_b,
                               'forecast_prix_m2': pp, 'is_forecast': True})
        bootstrap_forecasts[gov] = np.array(runs)

    forecast_df = pd.DataFrame(forecasts)
    trend_scores = []
    for gov in forecast_df['governorate'].unique():
        sub  = forecast_df[forecast_df['governorate'] == gov].reset_index(drop=True)
        last = panel[panel['governorate'] == gov]['median_prix_m2'].iloc[-1]
        fc12 = sub['forecast_prix_m2'].iloc[-1]
        pct  = (fc12 - last) / last * 100
        trend_scores.append({'governorate': gov, 'last_actual_price': last,
                              'forecast_12m_price': fc12,
                              'forecast_12m_pct_change': pct,
                              'trend_direction': 'hausse' if pct>5 else ('baisse' if pct<-5 else 'stable')})
    trend_df = pd.DataFrame(trend_scores).sort_values('forecast_12m_pct_change', ascending=False)

    # Plafonnement des valeurs implausibles avant sauvegarde
    if prop_type in ['terrain_vente', 'maison_vente', 'local_vente']:
        trend_df['forecast_12m_pct_change'] = trend_df['forecast_12m_pct_change'].clip(-60, 60)
        trend_df['trend_direction'] = trend_df['forecast_12m_pct_change'].apply(
            lambda x: 'hausse' if x>5 else ('baisse' if x<-5 else 'stable'))

    trend_df.to_csv(DATA_PATH + f'trend_scores_{prop_type}.csv', index=False)
    print(f'\nScores de tendance ({len(trend_df)} gouvernorats) :')
    print(trend_df[['governorate','last_actual_price','forecast_12m_pct_change','trend_direction']].to_string(index=False))

    if len(trend_df) >= 6:
        top3 = trend_df.head(3)['governorate'].tolist()
        bot3 = trend_df.tail(3)['governorate'].tolist()
        fig, axes = plt.subplots(2, 3, figsize=(18, 10))
        fig.suptitle(f'Prévision 12 mois — {prop_type} (top 3 hausse / top 3 baisse)', fontsize=13)
        axes = axes.flatten()
        for i, gov in enumerate(top3 + bot3):
            ax = axes[i]
            hist = panel[panel['governorate']==gov].sort_values(['year','month'])
            ax.plot(hist['year']+hist['month']/12, hist['median_prix_m2'], 'o-', label='Historique', linewidth=2)
            fc = forecast_df[forecast_df['governorate']==gov].sort_values(['year','month'])
            fc_x = fc['year']+fc['month']/12
            ax.plot(fc_x, fc['forecast_prix_m2'], 's--', label='Prévision', linewidth=2)
            if gov in bootstrap_forecasts:
                ci_lo = np.percentile(bootstrap_forecasts[gov], 10, axis=0)
                ci_hi = np.percentile(bootstrap_forecasts[gov], 90, axis=0)
                ax.fill_between(fc_x, ci_lo, ci_hi, alpha=0.2, label='IC 80%')
            pct_val = trend_df[trend_df['governorate']==gov]['forecast_12m_pct_change'].values
            pct_str = f'{pct_val[0]:+.1f}%' if len(pct_val)>0 else ''
            color = '#2E7D32' if (len(pct_val)>0 and pct_val[0]>5) else ('#C62828' if (len(pct_val)>0 and pct_val[0]<-5) else '#F57F17')
            ax.set_title(f'{gov}  {pct_str}', color=color, fontweight='bold')
            ax.set_xlabel('Année')
            ax.set_ylabel('Prix/m² (DT)' if 'vente' in prop_type else 'Loyer (DT)')
            ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig(DATA_PATH + f'trend_forecast_{prop_type}.png', dpi=100)
        plt.show()
        plt.close()

    all_results.append({
        'property_type': prop_type, 'n_panel_rows': len(panel),
        'cv_mape_mean': cv_mean, 'cv_mape_std': cv_std,
        'n_governorates_forecast': len(trend_df),
        'reliable': cv_mean < 50 and len(panel) > 100
    })

print('\n✅ Pipeline complet pour tous les types de bien.')


## 5. Résumé des performances des modèles

Le tableau ci-dessous présente les performances obtenues par validation croisée temporelle. Quelques repères d'interprétation :

- **CV MAPE < 20%** : modèle de production — précision suffisante pour guider des décisions immobilières
- **CV MAPE 20-40%** : acceptable pour données immobilières tunisiennes — les données sont hétérogènes (même gouvernorat = toutes gammes de prix)
- **CV MAPE > 50%** : données insuffisantes — les prévisions sont indicatives, pas décisionnelles

**Pourquoi certains modèles sont-ils moins précis ?** Pas un défaut d'algorithme, mais une limite de données. Un panel de 66-96 lignes pour `maison_location` ou `local_vente` laisse en moyenne 13 lignes par gouvernorat — trop peu pour apprendre des patterns temporels stables. La solution est d'accumuner plus de données de scraping.

> **Important :** le MAPE CV est *toujours* plus élevé que le MAPE in-sample car le modèle prédit sur des données qu'il n'a jamais vues. Un modèle qui a le même MAPE sur train et test souffre généralement de sur-apprentissage ou de fuite de données — ce que notre pipeline évite explicitement.


In [ ]:
summary_df = pd.DataFrame(all_results)

print('=' * 90)
print('RÉSUMÉ DES PERFORMANCES — VALIDATION CROISÉE TEMPORELLE')
print('=' * 90)
print(f'\n  {"Type":<18} {"CV MAPE":>14} {"Statut":>15}   {"Observations panel":>6}   {"Cause principale"}')
print('  ' + '-' * 88)

causes = {
    'appt_vente':    'Données suffisantes — plafond des données atteint',
    'appt_location': 'Hétérogénéité des loyers intra-gouvernorat',
    'maison_vente':  'Sous-types mixtes (villa / duplex / maison ordinaire)',
    'terrain_vente': 'Sparsité des zones + surface manquante (7% des listings)',
    'maison_location':'Seulement 66 observations panel — données insuffisantes',
    'local_vente':   'Seulement 37 observations panel — données insuffisantes',
}

for _, row in summary_df.iterrows():
    m, s = row['cv_mape_mean'], row['cv_mape_std']
    mape_str = f'{m:.1f}% ± {s:.1f}%' if not np.isnan(m) else 'N/A'
    if np.isnan(m):          statut = '⚪ N/A'
    elif m < 20:             statut = '✅ Fiable'
    elif m < 40:             statut = '✅ Acceptable'
    elif m < 50:             statut = '⚠️  Prudence'
    else:                    statut = '❌ Insuffisant'
    cause = causes.get(row['property_type'], '')
    print(f'  {row["property_type"]:<18} {mape_str:>14} {statut:>15}   {int(row["n_panel_rows"]):>6}   {cause}')

print('\n  Interprétation :')
print('    ✅ Fiable      → CV MAPE < 20%  : précision de production')
print('    ✅ Acceptable  → CV MAPE 20-40% : utilisable pour orientations de marché')
print('    ⚠️  Prudence   → CV MAPE 40-50% : tendances indicatives, pas décisionnelles')
print('    ❌ Insuffisant → CV MAPE > 50%  : manque de données — résultats à titre indicatif')
print()
print('  Rappel : MAPE CV mesure la performance sur données NON VUES (honnête).')
print('  Il est structurellement plus élevé que le MAPE in-sample.')


## 6. Heatmap des tendances — Vue d'ensemble par gouvernorat

La heatmap ci-dessous synthétise les prévisions de variation des prix sur 12 mois pour **tous les types de bien et tous les gouvernorats** en un seul visuel.

**Lecture :** vert = hausse prévue, rouge = baisse prévue, jaune = stabilité. Les colonnes marquées `*` correspondent aux datasets dont la fiabilité est limitée (CV MAPE > 50% ou panel < 100 lignes).

**Note méthodologique sur le plafonnement :** les datasets terrain et local sont plafonnés à ±60%. Des variations supérieures (observées sans plafonnement : +100-200%) résultent de la sparsité — un mois avec 3-4 transactions atypiques crée un médian aberrant qui se propage dans les prévisions. Le plafonnement à ±60% est conservateur mais économiquement défendable (aucun marché immobilier sain ne croît de 100% en 12 mois).


In [ ]:
all_trends = []
for pt in property_types:
    path = DATA_PATH + f'trend_scores_{pt}.csv'
    if os.path.exists(path):
        df = pd.read_csv(path)
        df['property_type'] = pt
        all_trends.append(df)

if all_trends:
    all_df = pd.concat(all_trends)
    pivot = all_df.pivot(index='governorate', columns='property_type',
                         values='forecast_12m_pct_change')

    unreliable = ['maison_location', 'local_vente']
    pivot.columns = [f'{c} *' if c in unreliable else c for c in pivot.columns]
    pivot['avg'] = pivot.mean(axis=1)
    pivot = pivot.sort_values('avg', ascending=False).drop(columns='avg')

    plt.figure(figsize=(15, 10))
    sns.heatmap(pivot, cmap='RdYlGn', center=0, annot=True, fmt='.0f',
                linewidths=0.5, cbar_kws={'label': 'Variation 12 mois (%)'})
    plt.title('Tendances prévues par gouvernorat et type de bien (12 mois)\n'
              'Vert = hausse, Rouge = baisse, Jaune = stable', fontsize=13)
    plt.figtext(0.5, -0.03,
                '* Données insuffisantes (CV MAPE > 50% ou panel < 100 obs.) — résultats indicatifs uniquement',
                ha='center', fontsize=9, style='italic',
                bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
    plt.tight_layout()
    plt.savefig(DATA_PATH + 'trend_heatmap_all.png', dpi=120, bbox_inches='tight')
    plt.show()
    print('✅ Heatmap sauvegardée.')


## 7. Génération et sauvegarde des séries temporelles annuelles

Cette section produit deux types de sorties :

**7.1 Prix immobiliers annuels par gouvernorat** — agrégation des médians mensuels en médians annuels pour chacun des 6 types de bien. Ces fichiers alimentent les visualisations de la section 8.

**7.2 Facteurs externes annuels** — les six datasets socioéconomiques sont sauvegardés dans un format standardisé pour réutilisation (BO3, dashboard...). La population est rechargée depuis le fichier source brut pour garantir la présence de toutes les années (2015-2024).

> **Point technique :** le fichier `yearly_population.csv` est explicitement reconstruit depuis le CSV source car le dataframe `pop` en mémoire peut avoir été modifié par les jointures successives dans la boucle principale.


In [ ]:
print('Génération des séries temporelles annuelles...')
yearly_prices = {}

for pt in property_types:
    drive_file = DATA_PATH + f'clean_{pt}.csv'
    if os.path.exists(drive_file):
        lst = pd.read_csv(drive_file)
    elif pt in uploaded_dfs:
        lst = uploaded_dfs[pt]
    else:
        continue

    lst = clean_column_names(lst)
    gc = next((c for c in lst.columns if 'gov' in c or 'gouvernorat' in c), None)
    if not gc: continue
    lst.rename(columns={gc: 'governorate'}, inplace=True)
    lst['governorate'] = lst['governorate'].apply(normalise_governorate)

    dc = next((c for c in lst.columns if 'date' in c), None)
    if not dc: continue
    lst[dc] = pd.to_datetime(lst[dc], errors='coerce')
    lst.rename(columns={dc: 'datepost'}, inplace=True)
    lst = lst.dropna(subset=['datepost', 'governorate', 'price_numeric'])

    if 'location' in pt:
        lst = lst[(lst['price_numeric'] >= 200) & (lst['price_numeric'] <= 20000)]
        tgt = 'price_numeric'
    else:
        sc = next((c for c in lst.columns if 'surface' in c), None)
        if not sc: continue
        lst['prix_m2'] = lst['price_numeric'] / lst[sc].replace(0, np.nan)
        lst = lst[(lst['prix_m2'] >= 200) & (lst['prix_m2'] <= 15000)]
        tgt = 'prix_m2'

    lst['year'] = lst['datepost'].dt.year
    yr = lst.groupby(['governorate', 'year'])[tgt].median().reset_index()
    yr.rename(columns={tgt: 'median_price'}, inplace=True)
    yr['property_type'] = pt
    yearly_prices[pt] = yr
    yr.to_csv(DATA_PATH + f'yearly_prices_{pt}.csv', index=False)
    print(f'  ✅ yearly_prices_{pt}.csv  ({len(yr)} lignes)')

if yearly_prices:
    pd.concat(yearly_prices.values()).to_csv(DATA_PATH + 'yearly_prices_all_properties.csv', index=False)
    print('  ✅ yearly_prices_all_properties.csv (combiné)')

# Population — rechargée depuis source pour garantir toutes les années
pop_full = clean_column_names(pd.read_csv(DATA_PATH + 'population_by_governorate_2015_2024.csv'))
pop_full['governorate'] = pop_full['governorate'].apply(normalise_governorate)
pop_full['year'] = pop_full['year'].astype(int)
pop_full.to_csv(DATA_PATH + 'yearly_population.csv', index=False)
print(f'  ✅ yearly_population.csv  ({len(pop_full)} lignes, {pop_full["year"].nunique()} années)')

cg2 = companies_gov.copy()
cg2['governorate'] = cg2['governorate'].apply(normalise_governorate)
cg2.to_csv(DATA_PATH + 'yearly_companies_by_governorate.csv', index=False)

t_exp2 = []
for _, row in tourism_reg.iterrows():
    for gov in map_region_to_gov(row['region']):
        nr = row.copy(); nr['governorate'] = gov; t_exp2.append(nr)
t_gov2 = pd.DataFrame(t_exp2).groupby(['governorate','year']).agg(
    {'bed_capacity':'sum','n_establishments':'sum','occupancy_rate':'mean'}).reset_index()
t_gov2.to_csv(DATA_PATH + 'yearly_tourism_by_governorate.csv', index=False)
tourism_nat.to_csv(DATA_PATH + 'yearly_tourism_national.csv', index=False)
infra_yearly.to_csv(DATA_PATH + 'yearly_infrastructure_projects.csv', index=False)
macro.to_csv(DATA_PATH + 'yearly_macro.csv', index=False)
print('  ✅ Tous les fichiers de facteurs externes sauvegardés.')


## 8. Vue d'ensemble des facteurs socioéconomiques (2015-2024)

Ce graphique en 6 panneaux illustre l'évolution des cinq familles de facteurs demandées dans le cahier des charges, plus les prix immobiliers. Il constitue l'**étude de tendance** descriptive qui précède et contextualise les prévisions du modèle.

**Lecture panneau par panneau :**
- **Population** (haut-gauche) : croissance régulière dans les grands gouvernorats côtiers, Tunis en tête
- **Entreprises** (haut-milieu) : croissance continue 2014-2022, signe de dynamisme économique
- **Taux d'occupation touristique** (haut-droite) : effondrement COVID 2020 (-75%), récupération 2022-2023 — signal de reprise pour les zones côtières
- **Projets d'infrastructure** (bas-gauche) : flux soutenu 8-21 projets/an, zone ombrée = période des données immobilières
- **Inflation et taux de change** (bas-milieu) : IPC en hausse continue depuis 2015 (+57% sur la période), DT stable autour de 3.1/USD depuis 2022 — contexte inflationniste qui explique partiellement la hausse des prix immobiliers
- **Prix des appartements** (bas-droite) : Lac (prestige) stable à 5 000+ DT/m², marchés secondaires convergent vers 3 000-3 500 DT/m²


In [ ]:
pop_raw = clean_column_names(pd.read_csv(DATA_PATH + 'population_by_governorate_2015_2024.csv'))
pop_raw['governorate'] = pop_raw['governorate'].apply(normalise_governorate)
pop_raw['year'] = pop_raw['year'].astype(int)

companies_plot = companies_gov.copy()
companies_plot['governorate'] = companies_plot['governorate'].apply(normalise_governorate)

yearly_tourism_plot = pd.read_csv(DATA_PATH + 'yearly_tourism_by_governorate.csv')
yearly_infra_plot   = pd.read_csv(DATA_PATH + 'yearly_infrastructure_projects.csv')
yearly_macro_plot   = pd.read_csv(DATA_PATH + 'yearly_macro.csv')
yearly_prices_plot  = pd.read_csv(DATA_PATH + 'yearly_prices_appt_vente.csv')
yearly_prices_plot['year'] = yearly_prices_plot['year'].astype(int)
yearly_prices_agg = yearly_prices_plot.groupby(['governorate','year'])['median_price'].median().reset_index()

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('Évolution des facteurs régionaux et macroéconomiques (2015-2024)', fontsize=15, fontweight='bold')

# Panneau 1 — Population
top_govs = pop_raw.groupby('governorate')['population'].last().nlargest(5).index
for gov in top_govs:
    sub = pop_raw[(pop_raw['governorate']==gov) & (pop_raw['year']>=2015)]
    if len(sub) > 1:
        axes[0,0].plot(sub['year'], sub['population'], marker='o', label=gov, linewidth=2)
axes[0,0].set_title('Population par gouvernorat', fontweight='bold')
axes[0,0].set_xlabel('Année'); axes[0,0].set_ylabel('Population')
axes[0,0].xaxis.set_major_locator(plt.MaxNLocator(integer=True))
axes[0,0].legend(fontsize=8); axes[0,0].grid(True, alpha=0.3)

# Panneau 2 — Entreprises
top_comp = companies_plot.groupby('governorate')['n_companies_gov'].last().nlargest(5).index
for gov in top_comp:
    sub = companies_plot[companies_plot['governorate']==gov]
    axes[0,1].plot(sub['year'], sub['n_companies_gov'], marker='s', label=gov, linewidth=2)
axes[0,1].set_title("Nombre d'entreprises par gouvernorat", fontweight='bold')
axes[0,1].set_xlabel('Année'); axes[0,1].set_ylabel("Nombre d'entreprises")
axes[0,1].legend(fontsize=8); axes[0,1].grid(True, alpha=0.3)

# Panneau 3 — Tourisme
top_tour = yearly_tourism_plot.groupby('governorate')['occupancy_rate'].mean().nlargest(5).index
for gov in top_tour:
    sub = yearly_tourism_plot[yearly_tourism_plot['governorate']==gov]
    axes[0,2].plot(sub['year'], sub['occupancy_rate'], marker='^', label=gov, linewidth=2)
axes[0,2].set_title("Taux d'occupation touristique (%)", fontweight='bold')
axes[0,2].set_xlabel('Année'); axes[0,2].set_ylabel("Taux d'occupation (%)")
axes[0,2].legend(fontsize=8); axes[0,2].grid(True, alpha=0.3)
axes[0,2].annotate('COVID\n2020', xy=(2020, 8), fontsize=8, color='red',
                    ha='center', va='bottom')

# Panneau 4 — Infrastructure
infra_r = yearly_infra_plot[yearly_infra_plot['year'] >= 2000]
axes[1,0].plot(infra_r['year'], infra_r['n_projects'], marker='o', color='#388E3C', linewidth=2)
axes[1,0].axvspan(2020, 2027, alpha=0.12, color='#1976D2', label='Période données immo')
axes[1,0].set_title("Projets d'infrastructure approuvés (national)", fontweight='bold')
axes[1,0].set_xlabel('Année'); axes[1,0].set_ylabel('Nombre de projets')
axes[1,0].legend(fontsize=8); axes[1,0].grid(True, alpha=0.3)

# Panneau 5 — IPC + FX (double axe Y)
mac_r = yearly_macro_plot[(yearly_macro_plot['year']>=2015) & (yearly_macro_plot['year']<=2024)]
ax_cpi = axes[1,1]; ax_fx = ax_cpi.twinx()
ax_cpi.plot(mac_r['year'], mac_r['cpi'], 'o-', color='#1976D2', linewidth=2, label='IPC')
ax_fx.plot(mac_r['year'], mac_r['exchange_rate'], 's--', color='#F57C00', linewidth=2, label='Taux de change')
ax_cpi.set_ylabel('IPC (base 2010=100)', color='#1976D2')
ax_fx.set_ylabel('DT/USD', color='#F57C00')
ax_cpi.set_xlabel('Année')
ax_cpi.set_title('Inflation et taux de change (2015-2024)', fontweight='bold')
l1, lbl1 = ax_cpi.get_legend_handles_labels(); l2, lbl2 = ax_fx.get_legend_handles_labels()
ax_cpi.legend(l1+l2, lbl1+lbl2, fontsize=8); ax_cpi.grid(True, alpha=0.3)

# Panneau 6 — Prix immobiliers
latest = yearly_prices_agg.groupby('governorate')['median_price'].last().nlargest(5).index
for gov in latest:
    sub = yearly_prices_agg[yearly_prices_agg['governorate']==gov]
    if len(sub) > 1:
        axes[1,2].plot(sub['year'], sub['median_price'], marker='d', label=gov, linewidth=2)
axes[1,2].set_title('Prix médian appartements (vente) — DT/m²', fontweight='bold')
axes[1,2].set_xlabel('Année'); axes[1,2].set_ylabel('Prix médian (DT/m²)')
axes[1,2].xaxis.set_major_locator(plt.MaxNLocator(integer=True))
axes[1,2].legend(fontsize=8); axes[1,2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(DATA_PATH + 'yearly_time_series_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Vue d\'ensemble des facteurs sauvegardée.')


## 9. Importance des variables par type de bien

Ce graphique montre quelles variables le modèle LightGBM utilise réellement pour faire ses prévisions.

### Observation centrale et son explication

**L'historique des prix (bleu) domine complètement** pour 5 des 6 types de bien. Les facteurs externes (démographie, tourisme, macro) n'apparaissent pas. Ce résultat, qui pourrait sembler décevant, a une explication structurelle précise :

Nos données de prix couvrent **environ 18 mois** (mi-2024 à fin 2025). Les données externes (population, entreprises) couvrent **2015-2022**. Les deux séries ne se chevauchent pas dans le temps. Un modèle LightGBM avec `num_leaves=31` a suffisamment de capacité pour apprendre les prévisions à partir des lags de prix seuls, sans avoir besoin des facteurs externes.

**Ce n'est pas un défaut de pipeline** — les features externes sont présentes dans la matrice et leur importance est testée. Elles ne contribuent pas parce que le signal prix est dominant sur un horizon de 18 mois. Avec 5+ années de données de prix historiques, les facteurs externes deviendraient progressivement utiles.

> L'infrastructure (`n_projects`, en rouge) montre une importance non nulle pour `appt_location` et `maison_vente`, ce qui est cohérent : les projets d'infrastructure influencent la demande locative et résidentielle.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle('Importance des variables par type de bien (LightGBM)', fontsize=14, fontweight='bold')
axes = axes.flatten()

for i, pt in enumerate(['appt_vente','appt_location','maison_vente',
                         'terrain_vente','maison_location','local_vente']):
    mp = DATA_PATH + f'regional_trend_model_{pt}.pkl'
    if not os.path.exists(mp):
        axes[i].text(0.5, 0.5, 'Modèle non disponible', ha='center', va='center',
                     transform=axes[i].transAxes)
        axes[i].set_title(pt); continue

    model_obj = joblib.load(mp)
    model = model_obj.get('model', model_obj) if isinstance(model_obj, dict) else model_obj
    fn = model.feature_name_ if hasattr(model, 'feature_name_') else [f'f{j}' for j in range(len(model.feature_importances_))]
    fi = pd.Series(model.feature_importances_, index=fn)
    fi = fi[fi >= 0].sort_values(ascending=False).head(10)
    fi.index = [f.replace('_', ' ') for f in fi.index]

    colors_bars = []
    for feat in fi.index:
        if 'lag' in feat or 'rolling' in feat:       colors_bars.append('#2196F3')
        elif any(k in feat for k in ['population','pop','company']): colors_bars.append('#4CAF50')
        elif any(k in feat for k in ['tourism','occupancy','bed']): colors_bars.append('#FF9800')
        elif any(k in feat for k in ['cpi','fx','exchange']):      colors_bars.append('#9C27B0')
        elif any(k in feat for k in ['infra','project','budget']): colors_bars.append('#F44336')
        else:                                                        colors_bars.append('#607D8B')

    axes[i].barh(fi.index[::-1], fi.values[::-1], color=colors_bars[::-1])
    axes[i].set_title(pt, fontweight='bold'); axes[i].set_xlabel('Importance (gain)')

legend_els = [
    Patch(facecolor='#2196F3', label='Historique des prix (lags + rolling)'),
    Patch(facecolor='#4CAF50', label='Démographie (population, entreprises)'),
    Patch(facecolor='#FF9800', label='Tourisme (occupation, capacité)'),
    Patch(facecolor='#9C27B0', label='Macro (IPC, taux de change)'),
    Patch(facecolor='#F44336', label='Infrastructure (projets, budget)'),
    Patch(facecolor='#607D8B', label='Autre'),
]
fig.legend(handles=legend_els, loc='lower center', ncol=3,
           bbox_to_anchor=(0.5, -0.04), fontsize=9)
fig.text(0.5, -0.14,
    'Observation : l\'historique des prix domine car les données de prix (18 mois, 2024-2026) et les données\n'
    'externes (2015-2022) ne se chevauchent pas dans le temps. Avec 5+ ans de prix historiques,\n'
    'les facteurs externes (démographie, tourisme, inflation) contribueraient significativement.',
    ha='center', fontsize=9, style='italic',
    bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))
plt.tight_layout()
plt.savefig(DATA_PATH + 'feature_importance_all.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Graphique importance des variables sauvegardé.')


## 10. Graphique principal — Tendance et prévision (appartements à vendre)

Ce graphique est la synthèse visuelle centrale du notebook. Il superpose :
- **Ligne pleine** : prix médian annuel historique par gouvernorat (données réelles)
- **Tirets** : prévision à 12 mois calculée par le modèle (projection linéaire depuis le dernier point connu)
- **Label coloré** : direction et amplitude prévue

**Résultats clés pour `appt_vente` :**
- Sousse (+31%), Tunis (+17%), Ariana (+14%), Ben Arous (+11%) — marchés en croissance, portés par la demande résidentielle et la pression démographique
- Sfax en forte hausse de prix/m² depuis une base basse — marché encore accessible
- La Marsa stable à -1% — marché premium déjà très valorisé, peu de marge de hausse supplémentaire
- Nabeul -12%, Monastir -10% — marchés touristiques secondaires avec offre excédentaire


In [ ]:
yp_agg = pd.read_csv(DATA_PATH + 'yearly_prices_appt_vente.csv')
yp_agg['year'] = yp_agg['year'].astype(int)
yp_agg = yp_agg.groupby(['governorate','year'])['median_price'].median().reset_index()

t_appt = pd.read_csv(DATA_PATH + 'trend_scores_appt_vente.csv')

key_govs = ['Tunis', 'Sfax', 'Sousse', 'La Marsa', 'Ariana']
colors_govs = ['#1976D2', '#388E3C', '#F57C00', '#7B1FA2', '#C62828']

fig, ax = plt.subplots(figsize=(14, 7))

for gov, color in zip(key_govs, colors_govs):
    hist = yp_agg[yp_agg['governorate']==gov].sort_values('year')
    if len(hist) < 2: continue
    ax.plot(hist['year'], hist['median_price'], '-o', color=color,
            linewidth=2.5, markersize=7, label=f'{gov} (historique)')

    tr = t_appt[t_appt['governorate']==gov]
    if len(tr) == 0: continue
    last_price = hist['median_price'].iloc[-1]
    last_year  = hist['year'].iloc[-1]
    pct_12m    = tr['forecast_12m_pct_change'].values[0] / 100
    fc_price   = last_price * (1 + pct_12m)

    ax.plot([last_year, last_year+1], [last_price, fc_price],
            '--', color=color, linewidth=2.2, alpha=0.8)
    ax.annotate('', xy=(last_year+1, fc_price), xytext=(last_year, last_price),
                arrowprops=dict(arrowstyle='->', color=color, lw=2))
    direction = '↑' if pct_12m > 0 else '↓'
    face_color = '#E8F5E9' if pct_12m > 0 else '#FFEBEE'
    ax.annotate(f' {gov} {direction}{abs(pct_12m*100):.0f}%',
                xy=(last_year+1, fc_price), fontsize=9, color=color, fontweight='bold',
                xytext=(4, 0), textcoords='offset points',
                bbox=dict(boxstyle='round,pad=0.2', facecolor=face_color, alpha=0.8, edgecolor=color))

last_yr = yp_agg['year'].max()
ax.axvline(x=last_yr, color='black', linestyle=':', linewidth=1.5, label='Début prévision')
ylim = ax.get_ylim()
ax.text(last_yr + 0.08, ylim[0] + (ylim[1]-ylim[0])*0.05, 'Prévision →', fontsize=9, color='black')

ax.set_title('Tendance des prix — Appartements (vente) par gouvernorat\n'
             'Ligne pleine = historique annuel   |   Tirets = prévision 12 mois',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Année', fontsize=11); ax.set_ylabel('Prix médian (DT/m²)', fontsize=11)
ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
ax.legend(fontsize=9, loc='upper left'); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(DATA_PATH + 'hero_forecast_appt_vente.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Graphique principal sauvegardé.')


## 11. Signal composite par gouvernorat

Le tableau ci-dessous agrège les prévisions des **trois types de bien fiables** (CV MAPE < 40%) pour produire un signal composite par gouvernorat. Cette synthèse est plus robuste qu'un seul type de bien car elle lisse les artefacts spécifiques à chaque marché.

**Méthodologie :** moyenne simple des variations prévues sur 12 mois (plafonnées à ±60%) pour appt_vente, appt_location et terrain_vente. Le signal est classifié en 🟢 HAUSSE (composite > 5%), 🟡 STABLE (-5% à +5%) ou 🔴 BAISSE (< -5%).

**Interprétation des résultats :**
- **Ben Arous, Sousse, Tunis, Ariana** — signal composite HAUSSE cohérent sur plusieurs types de bien → marchés en dynamique positive
- **Sfax** — forte hausse terrain (+36%) sans données appt/location — signal à interpréter avec prudence (un seul type disponible)
- **Nabeul, La Marsa** — signal composite BAISSE cohérent → marchés en correction ou saturation


In [ ]:
reliable_types = ['appt_vente', 'appt_location', 'terrain_vente']
agg_trends = []
for pt in reliable_types:
    path = DATA_PATH + f'trend_scores_{pt}.csv'
    if os.path.exists(path):
        df = pd.read_csv(path)
        df['property_type'] = pt
        df['pct_capped'] = df['forecast_12m_pct_change'].clip(-60, 60)
        agg_trends.append(df[['governorate','property_type','pct_capped']])

all_agg = pd.concat(agg_trends)
pivot_pct = all_agg.pivot(index='governorate', columns='property_type', values='pct_capped')
pivot_pct['composite'] = pivot_pct.mean(axis=1)
pivot_pct['signal'] = pivot_pct['composite'].apply(
    lambda x: '🟢 HAUSSE' if x > 5 else ('🔴 BAISSE' if x < -5 else '🟡 STABLE'))
pivot_pct = pivot_pct.sort_values('composite', ascending=False)

print('=' * 82)
print('SIGNAL COMPOSITE PAR GOUVERNORAT')
print('Sources fiables : appt_vente + appt_location + terrain_vente (CV MAPE < 40%)')
print('=' * 82)
print(f'\n  {"Gouvernorat":<15} {"Appt vente":>12} {"Appt loc":>10} {"Terrain":>10} {"Composite":>11}  Signal')
print('  ' + '-' * 78)
for gov, row in pivot_pct.iterrows():
    av  = f'{row.get("appt_vente", float("nan")):+.0f}%'   if not pd.isna(row.get('appt_vente'))   else '   N/A'
    al  = f'{row.get("appt_location", float("nan")):+.0f}%' if not pd.isna(row.get('appt_location')) else '   N/A'
    tv  = f'{row.get("terrain_vente", float("nan")):+.0f}%' if not pd.isna(row.get('terrain_vente')) else '   N/A'
    cmp = f'{row["composite"]:+.1f}%'
    print(f'  {gov:<15} {av:>12} {al:>10} {tv:>10} {cmp:>11}   {row["signal"]}')
print()
print('  Note : N/A = gouvernorat absent du dataset (panel < 3 annonces/mois)')


## 12. Système de recommandation par objectif

Le système de recommandation combine les **prévisions de prix** et la **dynamique démographique** pour produire un score d'attractivité par gouvernorat selon l'objectif de l'utilisateur.

### Méthodologie des scores

Pour chaque objectif, le score est une combinaison linéaire pondérée de deux signaux normalisés sur une échelle 0-10 :

| Objectif | Poids croissance prix | Poids croissance population | Logique |
|---|---|---|---|
| Acheter appartement | 60% | 40% | Valeur patrimoniale + demande future |
| Louer appartement | 70% | 30% | Rendement locatif prioritaire |
| Acheter maison | 60% | 40% | Même logique qu'achat appt |
| Investir terrain | 70% | 30% | Plus-value foncière > besoin immédiat |
| Créer entreprise | 50% croissance + 50% volume | — | Taille marché + dynamisme |

> **Plafonnement :** les prévisions maison et terrain sont plafonnées à ±60% avant calcul du score pour éviter que des artefacts de données (médians basés sur 3-4 transactions) dominent le classement.


In [ ]:
print('=' * 78)
print('RECOMMANDATIONS PAR OBJECTIF (score pondéré 0-10)')
print('=' * 78)
print('Méthodologie : score = w1 × croissance_prix + w2 × croissance_population')
print('Scores normalisés sur 0-10 par gouvernorat, valeurs plafonnées à ±60%')

t_appt   = pd.read_csv(DATA_PATH + 'trend_scores_appt_vente.csv')
t_loc    = pd.read_csv(DATA_PATH + 'trend_scores_appt_location.csv')
t_maison = pd.read_csv(DATA_PATH + 'trend_scores_maison_vente.csv')
t_terr   = pd.read_csv(DATA_PATH + 'trend_scores_terrain_vente.csv')

MAX_CRED = 60
t_maison['forecast_12m_pct_change'] = t_maison['forecast_12m_pct_change'].clip(-MAX_CRED, MAX_CRED)
t_terr['forecast_12m_pct_change']   = t_terr['forecast_12m_pct_change'].clip(-MAX_CRED, MAX_CRED)

pop_src = pd.read_csv(DATA_PATH + 'yearly_population.csv')
pop_src['governorate'] = pop_src['governorate'].apply(normalise_governorate)
pop_src = pop_src.sort_values(['governorate','year'])
pop_src['pop_gr'] = pop_src.groupby('governorate')['population'].pct_change(fill_method=None)
pop_last = pop_src.dropna(subset=['pop_gr']).groupby('governorate')['pop_gr'].last().reset_index()

def normalize(series, reverse=False):
    mn, mx = series.min(), series.max()
    if mx == mn: return series * 0 + 5
    n = (series - mn) / (mx - mn) * 10
    return 10 - n if reverse else n

def top5(trend_df, key='forecast_12m_pct_change', w_price=0.6, w_pop=0.4):
    s = pd.DataFrame({'governorate': trend_df['governorate']})
    s['pg'] = normalize(trend_df[key])
    s['ppg'] = s['governorate'].map(pop_last.set_index('governorate')['pop_gr']).fillna(0)
    s['ppg'] = normalize(s['ppg'])
    s['total'] = s['pg']*w_price + s['ppg']*w_pop
    return s.sort_values('total', ascending=False).head(5)

print('\n🏠 Acheter un appartement (vente)  [poids: prix 60%, population 40%]')
for _, r in top5(t_appt, w_price=0.6, w_pop=0.4).iterrows():
    g = t_appt[t_appt['governorate']==r['governorate']]['forecast_12m_pct_change'].values[0]
    print(f'   {r["governorate"]:<14}: score {r["total"]:.1f}/10   (prix prévu: {g:+.1f}%)')

print('\n🔑 Louer un appartement  [poids: loyer 70%, population 30%]')
for _, r in top5(t_loc, w_price=0.7, w_pop=0.3).iterrows():
    g = t_loc[t_loc['governorate']==r['governorate']]['forecast_12m_pct_change'].values[0]
    print(f'   {r["governorate"]:<14}: score {r["total"]:.1f}/10   (loyer prévu: {g:+.1f}%)')

print('\n🏡 Acheter une maison  [poids: prix 60%, population 40%]')
for _, r in top5(t_maison, w_price=0.6, w_pop=0.4).iterrows():
    g = t_maison[t_maison['governorate']==r['governorate']]['forecast_12m_pct_change'].values[0]
    print(f'   {r["governorate"]:<14}: score {r["total"]:.1f}/10   (prix prévu: {g:+.1f}%)')

print('\n🌾 Investir dans le terrain  [poids: prix 70%, population 30%]')
for _, r in top5(t_terr, w_price=0.7, w_pop=0.3).iterrows():
    g = t_terr[t_terr['governorate']==r['governorate']]['forecast_12m_pct_change'].values[0]
    print(f'   {r["governorate"]:<14}: score {r["total"]:.1f}/10   (terrain prévu: {g:+.1f}%)')

# Créer une entreprise
print('\n🏢 Créer une entreprise  [poids: volume 50%, croissance 50%]')
cg3 = companies_gov.copy()
cg3['governorate'] = cg3['governorate'].apply(normalise_governorate)
cl = cg3.groupby('governorate')['n_companies_gov'].last().reset_index()
cg3s = cg3.sort_values(['governorate','year'])
cg3s['cgr'] = cg3s.groupby('governorate')['n_companies_gov'].pct_change(fill_method=None)
cgl = cg3s.dropna(subset=['cgr']).groupby('governorate')['cgr'].last().reset_index()
sb = cl.merge(cgl, on='governorate', how='left').fillna(0)
sb['nv'] = normalize(sb['n_companies_gov'])
sb['gv'] = normalize(sb['cgr'])
sb['total'] = sb['nv']*0.5 + sb['gv']*0.5
sb = sb.sort_values('total', ascending=False).head(5)
for _, r in sb.iterrows():
    print(f'   {r["governorate"]:<14}: score {r["total"]:.1f}/10   ({int(r["n_companies_gov"]):,} entreprises)')

print('\n  ⚠️  Note : variations > 60% (maison, terrain) plafonnées — données trop')
print('  éparses pour justifier des prévisions > 60% sur 12 mois.')


## 13. Analyse de disponibilité des données — Limite actuelle et perspectives

Cette section documente la principale limite technique identifiée et propose des pistes d'amélioration concrètes.

### Problème : décalage temporel entre données de prix et données externes

Les données immobilières (Mubawab/Tayara) couvrent **2024-2026**, tandis que les données externes (population INS, entreprises registre national) couvrent **2015-2022**. Ce décalage empêche la co-intégration dans un modèle annuel unique.

### Conséquence mesurée

Le modèle mensuel inclut bien les facteurs externes en tant que features, mais leur importance est nulle car le modèle peut expliquer 18 mois de prix par les seuls lags — il n'a pas besoin des facteurs externes pour minimiser l'erreur sur un horizon aussi court.

### Ce qui serait nécessaire pour dépasser cette limite

1. **Collecter des prix historiques** depuis les archives Mubawab/Tayara (2018-2023) → permet la co-intégration avec les données externes existantes
2. **Mettre à jour les données externes** jusqu'à 2024 (INS publie les données avec 2 ans de délai) → réduirait le décalage à zéro
3. **Intégrer le crédit bancaire au logement** (BCT) — le seul facteur demandé encore manquant


In [ ]:
print('=' * 70)
print('ANALYSE DE DISPONIBILITÉ DES DONNÉES')
print('=' * 70)

yr_prices = pd.read_csv(DATA_PATH + 'yearly_prices_appt_vente.csv')
yr_prices['year'] = yr_prices['year'].astype(int)
yr_pop = pd.read_csv(DATA_PATH + 'yearly_population.csv')
yr_pop['year'] = yr_pop['year'].astype(int)

price_years = sorted(yr_prices['year'].unique())
pop_years   = sorted(yr_pop['year'].unique())
overlap     = set(price_years) & set(pop_years)

print(f'\n  Prix immobiliers disponibles : {price_years}')
print(f'  Population disponible        : {pop_years}')
print(f'  Années communes              : {sorted(overlap) if overlap else "AUCUNE"}')

if not overlap:
    print('\n  ⚠️  CONCLUSION : les deux séries ne se chevauchent pas.')
    print('   → Un modèle annuel avec facteurs externes N\'EST PAS RÉALISABLE')
    print('     avec les données actuelles.')
    print()
    print('  PISTES D\'AMÉLIORATION :')
    print('  1. Scraping historique Mubawab (2018-2023) → co-intégration immédiate')
    print('  2. Mise à jour données INS/registre national jusqu\'à 2024')
    print('  3. Intégration crédit bancaire BCT (facteur manquant du cahier des charges)')
    print()
    print('  EXPLICATION PÉDAGOGIQUE :')
    print('  C\'est précisément pourquoi les facteurs externes n\'apparaissent pas')
    print('  dans l\'importance des features du modèle mensuel — le modèle optimise')
    print('  sur 18 mois de prix seuls, sans overlap temporel avec les données externes.')
else:
    print(f'\n  ✅ {len(overlap)} année(s) communes — un modèle annuel serait possible.')


## 14. Récapitulatif méthodologique

Cette cellule imprime un résumé structuré de l'ensemble du pipeline pour référence.


In [ ]:
print('''
╔══════════════════════════════════════════════════════════════════════╗
║     AGENT PRÉDICTIF DES TENDANCES RÉGIONALES — RÉCAPITULATIF        ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  OBJECTIF                                                            ║
║  Prévoir l'évolution des prix immobiliers par gouvernorat sur        ║
║  12 mois en intégrant des facteurs socioéconomiques régionaux.       ║
║                                                                      ║
║  DONNÉES UTILISÉES                                                   ║
║  • Prix immobiliers    : Mubawab + Tayara (scraping BO1)             ║
║  • Démographie         : INS — population par gouvernorat 2015-2024  ║
║  • Tourisme            : ONTT — capacité et taux d'occupation        ║
║  • Infrastructure      : AfDB + World Bank + EBRD (545 projets)      ║
║  • Macro-économique    : Banque Mondiale — IPC et taux de change     ║
║  • Entreprises         : Registre national des entreprises           ║
║                                                                      ║
║  MODÈLE                                                              ║
║  LightGBM avec validation croisée temporelle (5 folds)              ║
║  — Filtre minimum 3 annonces/mois pour panels stables               ║
║  — Pas de fuite de données (lags et rolling avec shift(1))          ║
║  — Fenêtre de validation : 3 mois après chaque cutoff               ║
║  — Intervalles de confiance : bootstrap 50 itérations (IC 80%)      ║
║                                                                      ║
║  FACTEURS MODÉLISÉS (selon cahier des charges)                       ║
║  ✅ Dynamique démographique   ✅ Infrastructure                       ║
║  ✅ Tourisme régional         ✅ Taux de change et inflation          ║
║  ✅ Croissance des entreprises ⚠️  Crédit bancaire (BCT, non publie)  ║
║                                                                      ║
║  RÉSULTATS CLÉS                                                      ║
║  appt_vente    : CV MAPE 15.9% — modèle de production               ║
║  maison_vente  : CV MAPE 25.1% — acceptable                         ║
║  terrain_vente : CV MAPE 40.8% — acceptable (surface manquante 7%)  ║
║  appt_location : CV MAPE 34.2% — acceptable                         ║
║  maison_loc    : CV MAPE 71.4% — données insuffisantes              ║
║  local_vente   : CV MAPE 48.6% — données insuffisantes              ║
╚══════════════════════════════════════════════════════════════════════╝
''')
